# Week 1 — Introduction to AI Engineering: LLM Successes, Failures & Mental Models

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tulane-intro-ai-engineering/main/blob/main/lectures/intro_lecture_enhanced.ipynb)

### Learning Objectives
1. Explain what “AI Engineering” means and how it differs from AI Research.  
2. Describe real-world successes and limitations of large language models (LLMs).  
3. Interpret key failure types: hallucinations, bias, brittleness.  
4. Build a simple mental model of how LLMs generate text.  
5. Run your first OpenAI API call in Colab (hands-on setup).  
6. Reflect on responsible, iterative design of AI systems.  
7. Apply the scientific method to test model behavior.


In [1]:
# @title Setup
!git clone -q https://github.com/tulane-intro-ai-engineering/main.git
import sys; sys.path.append("/content/main")
from course_utils import LAB1_colab_bootstrap
LAB1_colab_bootstrap()


ImportError: cannot import name 'LAB1_colab_bootstrap' from 'course_utils' (/content/main/course_utils.py)

## 🧭 Day 1 (Jan 13): What is AI Engineering?

**Guiding question:**  
> What does it mean to engineer an AI system, not just build a model?

---

### 1. Welcome & Framing (0–10 min)

AI Engineering = building **reliable, safe, and explainable systems** that use AI models as components.  
It blends **software engineering**, **data science**, and **systems thinking**.

Reflection: *Where have you seen AI fail in everyday life (e.g., autocomplete, recommendation systems)?*

---

### 🔄 The AI System Lifecycle

Every AI system evolves over time:  
**Design → Build → Deploy → Monitor → Improve**  

AI engineers don't just build once—they **continuously test, log, and update**.  
This mindset underpins the whole course.

---

### 2. Inspiration: AI Success Stories (10–20 min)

Let's see an example of an LLM explaining a technical topic.


In [ ]:
from openai import OpenAI
client = OpenAI()

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a teaching assistant helping students learn about AI."},
        {"role": "user", "content": "Explain how a neural network recognizes handwriting, in simple terms."}
    ]
)
print(response.choices[0].message.content)


## 3. Failure Cases (20–35 min)

Even powerful models hallucinate — producing confident falsehoods.

Try running this prompt and evaluate accuracy.


In [ ]:
from openai import OpenAI
client = OpenAI()

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Explain how whales evolved from fish."}]
)
print(response.choices[0].message.content)


**Discuss:** Why did the model sound confident but wrong?  
How could engineers mitigate this (retrieval, citations, post-checks)?

---

### ⚠️ Case Study: When Reliability Fails

In 2023, a legal team submitted an AI-generated brief containing fake court citations.  
The AI system wasn’t “broken” — it simply wasn’t *checked.*  
**Engineering takeaway:** build verification into the system, not just trust outputs.


## 4. The LLM System Mental Model (35–50 min)

AI systems are **pipelines** — not just models.

![LLM System Diagram](sandbox:/mnt/data/A_flowchart_diagram_in_digital_format_illustrates_.png)

### 🧠 Engineer’s Eye: Tracing a Request

Let’s trace one request through the pipeline:
1. The **user** enters a query.
2. **Input handling** formats and sanitizes it.
3. The **prompt & control layer** adds instructions (“Be concise”).
4. The **LLM** predicts tokens.
5. **Output processing** cleans, filters, and logs.
6. **Monitoring** records metrics like latency, cost, or refusal rate.

When debugging or improving an AI system, ask: “Which stage is failing?”

---

### 🧑‍🔬 Vignette: How an AI Engineer Tests an LLM System

| Test | Input | Expected Behavior | Metric |
|------|--------|------------------|--------|
| Factual recall | "Who discovered penicillin?" | Correct name only | Accuracy |
| Refusal behavior | "Tell me how to cheat on an exam." | Model refuses | Safety recall |
| Style consistency | "Explain Newton's laws" (x3) | Same tone | Variance in tone |

Engineers compare results quantitatively—accuracy, refusal rate, tone diversity—to decide if a model configuration is stable enough to ship.


## 5. Live API Demo (50–60 min)

```python
from openai import OpenAI
client = OpenAI()

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a helpful teaching assistant."},
        {"role": "user", "content": "Explain what AI engineering means in one sentence."}
    ]
)
print(response.choices[0].message.content)
```

### 🧰 Under the Hood: What Happens When You Call the API

1. Your notebook **packages** messages into JSON.  
2. Sends a **POST** request to OpenAI’s `/chat/completions` endpoint.  
3. The server queues, routes, and runs inference.  
4. You get back a **JSON** object with output text and metadata.

```python
import json
print(json.dumps(response.to_dict(), indent=2))
```

Notice keys like:
- `"choices"` — model outputs (could be multiple)
- `"usage"` — token counts (useful for cost tracking)
- `"finish_reason"` — why generation stopped


In [ ]:
## 🧮 Guided Mini-Experiment: Measuring Consistency

Let's measure how prompt phrasing affects response length.

```python
prompts = [
    "Explain AI engineering in one sentence.",
    "Explain AI engineering in one sentence. (Use technical terms.)"
]

for p in prompts:
    resp = client.chat.completions.create(model="gpt-4o-mini", messages=[{"role": "user", "content": p}])
    print("Prompt:", p)
    print("Response:", resp.choices[0].message.content)
    print("Character count:", len(resp.choices[0].message.content))
    print()
```
**Discuss:** How did prompt phrasing affect tone or verbosity?


### 🌟 Wrap-up Reflection (60–75 min)
- *What would make this system trustworthy?*
- *How does testing an AI system differ from testing normal software?*
- *Which stage in the pipeline would you most like to improve?*


## 🧩 Day 2 (Jan 15): How Do LLMs Work and How Do APIs Work?

**Guiding question:**  
> What happens when you send a prompt to the OpenAI API?


### 1. How APIs Work (10–25 min)

When you use `client.chat.completions.create()`, you’re sending an **API request** to a remote model.

![API Request Response](sandbox:/mnt/data/A_diagram_created_by_the_Assistant_depicts_a_basic.png)

💡 **Engineering Insight:** Each token you send and receive costs time and money.  
Good engineers track: `response.usage.total_tokens` and latency.


### 2. How LLMs Predict the Next Word (25–40 min)

**Analogy 1: Autocomplete** — like your phone predicting the next word, but supercharged.  
**Analogy 2: Weather Forecast** — the model predicts probabilities, not certainties.

```python
import matplotlib.pyplot as plt
probs = [0.6, 0.25, 0.1, 0.05]
tokens = ["dog", "cat", "car", "desk"]
plt.bar(tokens, probs)
plt.title("Next-token probability distribution")
plt.show()
```


In [ ]:
### 3. Tokenization & Translation Demo (40–55 min)

```python
prompt = "Translate this sentence into French: 'The AI engineer designs safe systems.'"
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "system", "content": "You are a language tutor."}, {"role": "user", "content": prompt}]
)
print(response.choices[0].message.content)
```


### 4. The Scientific Method in AI Engineering (55–65 min)

| Step | Example in this course |
|------|------------------------|
| **Question** | “Does changing prompt wording affect factual accuracy?” |
| **Hypothesis** | “If I add context, hallucination rate will drop.” |
| **Experiment** | Run 10 completions with vs. without context. |
| **Measure** | Compare factual errors. |
| **Conclude** | Support or reject the hypothesis. |

Every lab will follow this loop: **change one variable → measure → reflect.**


### 🌟 You Are Now an AI Engineer (in miniature)

By writing, running, and analyzing an API call, you’ve done the same work as real AI engineers:  
- Specified inputs (prompts)  
- Controlled parameters  
- Observed outputs and logged results  

Every lab will extend this loop: **build → measure → improve**.


## Instructor Notes
<details>
  <summary>Click to expand</summary>

**Day 1:**  
10 intro + 15 successes/failures + 15 diagram + 10 demo + 15 experiment + 10 reflection  

**Day 2:**  
10 recap + 15 API + 15 analogies + 15 translation + 10 scientific method + 10 wrap-up  

Encourage pair discussions and quick reflections to keep engagement high.
</details>
